# Load Dataset

In [1]:
################################################################################
# Load dataset and split it into training and test set
################################################################################

import pandas as pd
import os
from tabulate import tabulate

dataset_name = "wustl-iiot"
sample_size = 100000

# Load dateset
data_path = os.path.join(
    os.path.expanduser("~"),
    "Documents", "Projects", "RAG Paper",
    "data", "wustl-iiot", "wustl-iiot-population.csv"
)

df = pd.read_csv(data_path, low_memory=False)

# Split dataset according to attack type
normal_df = df[df['Target'] == 0]
attack_df = df[df['Target'] == 1]

# Drop columns
normal_df = normal_df.drop(columns=['Target', 'Traffic'])
attack_df = attack_df.drop(columns=['Target', 'Traffic'])

# Split dataset into training and test set
normal_df_train = normal_df.sample(frac=0.8, random_state=42)
normal_df_test = normal_df.drop(normal_df_train.index)
attack_df_train = attack_df.sample(frac=0.8, random_state=42)
attack_df_test = attack_df.drop(attack_df_train.index)

# Print dataset sizes in a table
data = [
    ["Normal", normal_df.shape[0], normal_df_train.shape[0], normal_df_test.shape[0]],
    ["Attack", attack_df.shape[0], attack_df_train.shape[0], attack_df_test.shape[0]]
]
print(tabulate(data, headers=["Atack type", "Total", "Train", "Test"], tablefmt="grid"))

+--------------+---------+---------+--------+
| Atack type   |   Total |   Train |   Test |
+==============+=========+=========+========+
| Normal       | 1107448 |  885958 | 221490 |
+--------------+---------+---------+--------+
| Attack       |   87016 |   69613 |  17403 |
+--------------+---------+---------+--------+


# Feature Importance

In [11]:
################################################################################
# Generate Feature Importance
################################################################################

import os
import dotenv
import time
import numpy as np
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from sklearn.preprocessing import normalize

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Output top 10 important features that can be used to filter an entry as either normal or attack.
Output only in the Python list structure.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```

Example output:
['feature1', 'feature2', 'feature3', ..., 'feature10']
"""

prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
chain = prompt | llm
train_set_size = sample_size

# Helper function for fallback sampling
def _sample_via_dataframe(df, n):
    """Find n rows closest to mean using cosine similarity."""
    # Select only numeric columns
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        # If no numeric columns, just return first n rows as strings
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]

# Try loading from vector store, fallback to dataframe
try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings, 
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")
    
except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

normal_entries = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in normal_documents]

attack_entries = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in attack_documents]

completions = []
for i in range(10):
    completion = chain.invoke({
        "normal_entries": json.dumps(normal_entries),
        "attack_entries": json.dumps(attack_entries)
    })
    completions.append(completion.content)
    print(completion.content)
    time.sleep(10)

with open(f"results/feature-importance-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write("\n".join(completions))

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'SrcRate', 'Dur', 'TotBytes', 'dTtl']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'SrcRate', 'Dur', 'TotBytes', 'dTtl']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'SrcRate', 'Dur', 'TotAppByte', 'dTtl']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'Dur', 'TotBytes', 'dTtl', 'SrcRate']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'Dur', 'TotBytes', 'dTtl', 'TotAppByte']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'Dur', 'TotBytes', 'dTtl', 'SrcRate']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', 'SrcRate', 'Dur', 'TotBytes', 'dTtl']
```
```python
['DstPkts', 'DstBytes', 'DstLoad', 'DstRate', 'pLoss', 'SrcLoad', '

# Prediction

In [12]:
################################################################################
# Generate Rules with transposed data
################################################################################

import os
import dotenv
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize
import numpy as np
import uuid

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate 5 simple and deterministic rules for top 5 important features to filter an entry as either normal or attack. 
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
chain = prompt | llm
train_set_size = sample_size

# Helper function for fallback sampling
def _sample_via_dataframe(df, n):
    """Find n rows closest to mean using cosine similarity."""
    # Select only numeric columns
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        # If no numeric columns, just return first n rows as strings
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]

# Try loading from vector store, fallback to dataframe
try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings, 
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]
    else:
        raise ValueError("No attack vectors found")

except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, 10)
    attack_documents = _sample_via_dataframe(attack_df_train, 10)

normal_entries = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in normal_documents]

attack_entries = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in attack_documents]

completion = chain.invoke({
    "normal_entries": json.dumps(normal_entries),
    "attack_entries": json.dumps(attack_entries)
})

print(completion.content)

id = str(uuid.uuid4())
with open(f"results/llm/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"{completion.content}\n")

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...
```json
{
  "DstPkts": "if DstPkts == 0 then ATTACK else NORMAL",
  "pLoss": "if pLoss > 30 then ATTACK else NORMAL",
  "DstBytes": "if DstBytes == 0 then ATTACK else NORMAL",
  "Dur": "if Dur < 0.00001 then ATTACK else NORMAL",
  "dTtl": "if dTtl == 0 then ATTACK else NORMAL"
}
```


In [13]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("normal" if dataset.iloc[i]['SrcAddr'] == "192.168.0.20" else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['Dport'] == 502 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['SrcPkts'] == 10 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['DstPkts'] == 8 else "attack")
        predicted_attack_types.append("normal" if dataset.iloc[i]['TotBytes'] == 1152 else "attack")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred, digits=4)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/llm/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}\n")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 17403/17403 [00:01<00:00, 11576.09it/s]


              precision    recall  f1-score   support

      attack     0.3222    1.0000    0.4873     17403
      normal     1.0000    0.8347    0.9099    221490

    accuracy                         0.8467    238893
   macro avg     0.6611    0.9173    0.6986    238893
weighted avg     0.9506    0.8467    0.8791    238893

[[ 17403      0]
 [ 36613 184877]]


# Feedback Loop

In [2]:
################################################################################
# Prompt Template
################################################################################
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

system_message = ("system",
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
"""
)
human_message = ("user",
"""
Analyze the following network data and generate rules for the top 5 important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

# Invoke prompt
# prompt.invoke({"k": 5, "normal_entries": normal_entries, "attack_entries": attack_entries, "msgs": []})

In [3]:
################################################################################
# Define evaluate_rule tool
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = True
operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq, '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value: Annotated[str, "Value"], 
    op: Annotated[str, "Operator"]
) -> bool:
    """Evaluate the rule and return the macro f1-score."""
    try:
        value = float(value)
    except ValueError:
        value
    datasets = {"normal": normal_df_train, "attack": attack_df_train}
    y_pred = []
    y_true = []
    if op in operations:
        for attack_type, dataset in datasets.items():
            test_set_size = dataset.shape[0]
            for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries...", disable=not show_progress):
                y_true.append(attack_type)
                y_pred.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
        c_report = classification_report(y_true, y_pred, digits=4, output_dict=True)
        return c_report['macro avg']['f1-score']
    else:
        raise ValueError(f"Unsupported operator: {op}")

# Invoke tool
# print(evaluate_rule.invoke({"feature_name": "flow_duration", "value": "1", "op": "<"}))

In [4]:
################################################################################
# Initialize LLM
################################################################################

import os
import dotenv
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

model_name = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=model_name, temperature=0.1)
# model_name = "gemini-1.5-pro"
# llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)
# model_name = "claude-3-opus-20240229"
# llm = ChatAnthropic(model=model_name, temperature=0.0)

llm_with_tool = llm.bind_tools([evaluate_rule])

In [5]:
################################################################################
# Setup vector store
################################################################################

import json
import numpy as np
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize

train_set_size = sample_size
n_results = 10

# Helper function for fallback sampling
def _sample_via_dataframe(df, n):
    """Find n rows closest to mean using cosine similarity."""
    # Select only numeric columns
    df_numeric = df.select_dtypes(include=[np.number])
    if df_numeric.shape[1] == 0:
        # If no numeric columns, just return first n rows as strings
        return [str(df.iloc[i].to_list()) for i in range(min(n, len(df)))]
    X = df_numeric.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]

# Try loading from vector store, fallback to dataframe
try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings, 
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

    normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
    if normal_vectors is not None and len(normal_vectors) > 0:
        normal_mean_vector = np.mean(np.array(normal_vectors), axis=0).tolist()
        normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=n_results)['documents'][0]
    else:
        raise ValueError("No normal vectors found")

    attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
    if attack_vectors is not None and len(attack_vectors) > 0:
        attack_mean_vector = np.mean(np.array(attack_vectors), axis=0).tolist()
        attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=n_results)['documents'][0]
    else:
        raise ValueError("No attack vectors found")

except Exception as e:
    print(f"⚠  Vector store error: {e}")
    print("   Using dataframe-based sampling instead...")
    normal_documents = _sample_via_dataframe(normal_df_train, n_results)
    attack_documents = _sample_via_dataframe(attack_df_train, n_results)

normal_entries_dict = {}
for i, feature_name in enumerate(normal_df_train.columns.to_list()):
    normal_entries_dict[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in normal_documents]

attack_entries_dict = {}
for i, feature_name in enumerate(attack_df_train.columns.to_list()):
    attack_entries_dict[feature_name] = [json.loads(doc.replace("'", '"'))[i] if isinstance(doc, str) else doc[i] for doc in attack_documents]

⚠  Vector store error: No normal vectors found
   Using dataframe-based sampling instead...


In [6]:
from langchain_core.messages import HumanMessage

chain = prompt | llm_with_tool

n_repetitions = 5
context_window = 128000
show_progress = False

def get_initial_state():
  n = 0
  k = 5
  mean_f1s = 0
  max_f1s = 0
  n_max = 0
  token_usage = {}
  normal_entries = json.dumps(normal_entries_dict)
  attack_entries = json.dumps(attack_entries_dict)
  msgs = []
  return locals()

def extract_token_usage(ai_msg):
    """Extract token usage with safer fallback chain.
    
    First tries usage_metadata (newer langchain-anthropic),
    then response_metadata["token_usage"], then response_metadata["usage"].
    """
    # langchain-anthropic >= 0.2 uses usage_metadata
    if hasattr(ai_msg, "usage_metadata") and ai_msg.usage_metadata:
        meta = ai_msg.usage_metadata
        return {
            "prompt_tokens": meta.get("input_tokens", 0),
            "completion_tokens": meta.get("output_tokens", 0),
            "total_tokens": meta.get("total_tokens", meta.get("input_tokens", 0) + meta.get("output_tokens", 0)),
        }
    
    # Fallback to response_metadata
    token_usage = ai_msg.response_metadata.get("token_usage") or ai_msg.response_metadata.get("usage", {})
    return {
        "prompt_tokens": token_usage.get("prompt_tokens", token_usage.get("input_tokens", 0)),
        "completion_tokens": token_usage.get("completion_tokens", token_usage.get("output_tokens", 0)),
        "total_tokens": token_usage.get("total_tokens", 0),
    }

def extract_rules_from_tool_calls(tool_calls):
    """Extract rule definitions from tool calls.
    
    Handles both ToolCall objects (from ai_msg.tool_calls) and
    raw JSON dictionaries (from additional_kwargs).
    """
    rules = []
    for tool_call in tool_calls:
        try:
            # Handle ToolCall objects (have .args attribute)
            if hasattr(tool_call, 'args') and isinstance(tool_call.args, dict):
                args = tool_call.args
            # Handle raw JSON dictionaries
            elif isinstance(tool_call, dict) and "function" in tool_call:
                args = json.loads(tool_call["function"]["arguments"])
            else:
                continue
            
            rule = {
                "feature_name": str(args.get("feature_name", "")),
                "operator": str(args.get("op", "")),
                "value": str(args.get("value", ""))
            }
            rules.append(rule)
        except (KeyError, AttributeError, TypeError, json.JSONDecodeError) as e:
            print(f"Warning: Could not parse tool call: {e}")
            continue
    return rules

state = get_initial_state()
train_f1_scores = []
refinement_log = []

while state["n"] < n_repetitions:
    ai_msg = chain.invoke(state)
    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        tool_msg = evaluate_rule.invoke(tool_call)
        tool_msgs.append(tool_msg)
    state["mean_f1s"] = sum(float(msg.content) for msg in tool_msgs) / len(tool_msgs)
    
    # Extract selected rules and token usage
    selected_rules = extract_rules_from_tool_calls(ai_msg.tool_calls)
    token_info = extract_token_usage(ai_msg)
    
    # Log this round
    round_entry = {
        "round": state["n"] + 1,
        "macro_f1": state["mean_f1s"],
        "prompt_tokens": token_info["prompt_tokens"],
        "completion_tokens": token_info["completion_tokens"],
        "total_tokens": token_info["total_tokens"],
        "selected_rules": selected_rules
    }
    refinement_log.append(round_entry)
    
    human_msg = HumanMessage(f"The current mean f1-score for the generated rules is {state['mean_f1s']}. "
                             "If this mean f1-score is greater than the previous rounds, keep the better performing "
                             "rules and revise or replace only the underperforming ones (those with a score less than mean). "
                             "Otherwise, revise or replace any rules that have a score less than mean. "
                             f"Based on the feedback, generate exactly {state['k']} rules to filter attack entries and "
                             "make a tool call for each rule, ensuring that a tool call is made for every entry every time.")
    state["n"] += 1
    state["msgs"].extend([ai_msg, *tool_msgs, human_msg])
    train_f1_scores.append(state["mean_f1s"])
    state["max_f1s"] = state["mean_f1s"] if state["mean_f1s"] > state["max_f1s"] else state["max_f1s"]
    state["n_max"] = state["n"] if state["mean_f1s"] > state["max_f1s"] else state["n_max"]
    state["token_usage"] = extract_token_usage(ai_msg)
    print("Round:", state["n"], "Current mean f1-score:", state["mean_f1s"], "Token usage:", state["token_usage"])

print(train_f1_scores)

# Save refinement log to JSON file
import os
os.makedirs("results/llm", exist_ok=True)

refinement_summary = {
    "dataset": "wustl-iiot",
    "sample_size": sample_size,
    "seed": 42,
    "model_name": model_name,
    "n_rounds": len(refinement_log),
    "rounds": refinement_log,
    "train_f1_scores": train_f1_scores,
    "max_f1": state["max_f1s"],
    "max_f1_round": state["n_max"]
}

output_file = f"results/llm/policy-refinement-summary-{sample_size}-{model_name}.json"
with open(output_file, "w") as f:
    json.dump(refinement_summary, f, indent=2)

print(f"\nRefinement log saved to: {output_file}")

Round: 1 Current mean f1-score: 0.8880962902926882 Token usage: {'prompt_tokens': 5971, 'completion_tokens': 652, 'total_tokens': 6623}
Round: 2 Current mean f1-score: 0.8967994271128475 Token usage: {'prompt_tokens': 6921, 'completion_tokens': 701, 'total_tokens': 7622}
Round: 3 Current mean f1-score: 0.9003364207452789 Token usage: {'prompt_tokens': 7920, 'completion_tokens': 691, 'total_tokens': 8611}
Round: 4 Current mean f1-score: 0.8530135325461566 Token usage: {'prompt_tokens': 8909, 'completion_tokens': 707, 'total_tokens': 9616}
Round: 5 Current mean f1-score: 0.8892932532081137 Token usage: {'prompt_tokens': 9914, 'completion_tokens': 727, 'total_tokens': 10641}
[0.8880962902926882, 0.8967994271128475, 0.9003364207452789, 0.8530135325461566, 0.8892932532081137]

Refinement log saved to: results/llm/policy-refinement-summary-100000-claude-haiku-4-5-20251001.json


In [13]:
################################################################################
# Evaluate generated rules — per-round held-out test macro-F1
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
import operator
from statistics import mode
import json

operations = {"<": operator.lt, ">": operator.gt, "==": operator.eq,
              "<=": operator.le, ">=": operator.ge, "!=": operator.ne}

def _parse_args(tc):
    """Extract {feature_name, op, value} from any tool-call representation.

    LangChain ToolCall is a TypedDict with key 'args' (a plain dict).
    Older notebooks normalized to {"function": {"arguments": json_str}}.
    Some object-style wrappers expose .args as an attribute.
    """
    # TypedDict / plain dict with "args" key (LangChain native ToolCall)
    if isinstance(tc, dict) and "args" in tc:
        return tc["args"]
    # Normalized dict used in other notebooks
    if isinstance(tc, dict) and "function" in tc:
        return json.loads(tc["function"]["arguments"])
    # Object with .args attribute
    if hasattr(tc, "args"):
        return tc.args
    raise ValueError(f"Cannot extract args from tool call of type {type(tc)}: {tc}")

def evaluate_rules(tool_calls):
    """Apply combined majority-vote policy to the held-out test split."""
    datasets = {"normal": normal_df_test, "attack": attack_df_test}
    y_pred, y_true = [], []
    for attack_type, dataset in datasets.items():
        for i in range(len(dataset)):
            votes = []
            for tc in tool_calls:
                args = _parse_args(tc)
                feature_name = args["feature_name"]
                op = args["op"]
                value = args["value"]
                try:
                    value = float(value)
                except ValueError:
                    pass
                votes.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
            y_true.append(attack_type)
            y_pred.append(mode(votes))
    return classification_report(y_true, y_pred, digits=4, output_dict=True)

# Evaluate policy from each round on the held-out test split
for msg in state["msgs"]:
    if msg.type != "ai":
        continue
    tcs = msg.tool_calls
    if not tcs:
        continue
    for tc in tcs:
        args = _parse_args(tc)
        print(f"attack if {args['feature_name']} {args['op']} {args['value']} else normal")
    report = evaluate_rules(tcs)
    print(f"Test macro-F1: {report['macro avg']['f1-score']:.4f}")


attack if DstPkts == 0 else normal
attack if DstBytes == 0 else normal
attack if pLoss >= 33 else normal
attack if SrcLoad > 100000000 else normal
attack if DstLoad == 0 else normal
Test macro-F1: 0.9252
attack if DstPkts == 0 else normal
attack if DstBytes == 0 else normal
attack if pLoss >= 33 else normal
attack if Dur < 0.00001 else normal
attack if DstLoad == 0 else normal
Test macro-F1: 0.9252
attack if DstPkts == 0 else normal
attack if DstBytes == 0 else normal
attack if pLoss >= 33 else normal
attack if TotPkts == 2 else normal
attack if DstLoad == 0 else normal
Test macro-F1: 0.9252
attack if DstPkts == 0 else normal
attack if DstBytes == 0 else normal
attack if pLoss >= 33 else normal
attack if Loss == 1 else normal
attack if DstLoad == 0 else normal
Test macro-F1: 0.9252
attack if DstPkts == 0 else normal
attack if DstBytes == 0 else normal
attack if pLoss >= 33 else normal
attack if SrcRate > 100000 else normal
attack if DstLoad == 0 else normal
Test macro-F1: 0.9252


In [25]:
################################################################################
# Evaluate generated rules for efficiency
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from tabulate import tabulate
from statistics import mode
import time
import warnings
import pandas as pd
import os

warnings.filterwarnings("ignore")

sample_size = 100000

# Load dataset, falling back to the already-loaded dataframe when the sample file is absent
sample_path = os.path.join(os.getcwd(), f"data/sample-{sample_size}-2.csv")
if os.path.exists(sample_path):
    df = pd.read_csv(sample_path)
elif "df" in globals():
    df = df.copy()
else:
    fallback_path = os.path.join(
        os.path.expanduser("~"),
        "Documents", "Projects", "RAG Paper",
        "data", "wustl-iiot", "wustl-iiot-population.csv"
    )
    df = pd.read_csv(fallback_path, low_memory=False)

# Encode categorical columns
label_encoder = LabelEncoder()
categorical_columns = df.select_dtypes(include=['object']).columns
for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])

# Split dataset according to attack type
normal_df = df[df['Target'] == 0]
attack_df = df[df['Target'] == 1]

# Split dataset into training and test set
normal_df_train = normal_df.sample(frac=0.8, random_state=42)
normal_df_test = normal_df.drop(normal_df_train.index)
attack_df_train = attack_df.sample(frac=0.8, random_state=42)
attack_df_test = attack_df.drop(attack_df_train.index)

X_train = pd.concat([normal_df_train, attack_df_train]).drop(columns=['Target', 'Traffic'])
y_train = pd.concat([normal_df_train, attack_df_train])['Target']
X_test = pd.concat([normal_df_test, attack_df_test]).drop(columns=['Target', 'Traffic'])
y_test = pd.concat([normal_df_test, attack_df_test])['Target']

# Create instances of ML models
model_dt = DecisionTreeClassifier()
model_rf = RandomForestClassifier()

# Fit the models to the training data
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

# Predict the labels for the test data
y_true = y_test

elapsed_times_dt = []
elapsed_times_rf = []
elapsed_times_llm = []
y_pred_dt = []
y_pred_rf = []
y_pred_llm = []
for i in range(len(X_test)):
    # Predict using DT
    start = time.time()
    y_pred_dt.append(model_dt.predict([X_test.iloc[i]]))
    end = time.time()
    elapsed_times_dt.append(end - start)

    # Predict using RF
    start = time.time()
    y_pred_rf.append(model_rf.predict([X_test.iloc[i]]))
    end = time.time()
    elapsed_times_rf.append(end - start)
    
    # Predict using LLM
    start = time.time()
    row = X_test.iloc[i]
    # conditions = [
    #     row['SrcAddr'] != "192.168.0.20",
    #     row['SrcRate'] == 1000000,
    #     row['DstRate'] == 0,
    #     row['DstPkts'] == 0,
    #     row['Rate'] == 1000000
    # ]
    conditions = [
        row['SrcAddr'] != "192.168.0.20",
        row['TotPkts'] == 2,
        row['DstRate'] == 0,
        row['DstPkts'] == 0,
        row['DstLoad'] == 0
    ]
    predicted_attack_types = ["attack" if condition else "normal" for condition in conditions]
    y_pred_llm.append(mode(predicted_attack_types))
    end = time.time()
    elapsed_times_llm.append(end - start)

print(f"DT time taken: {sum(elapsed_times_dt)/len(X_test)}")
print(classification_report(y_true, y_pred_dt, digits=4, output_dict=False))
print(confusion_matrix(y_true, y_pred_dt))
print("\n")

print(f"RF time taken: {sum(elapsed_times_rf)/len(X_test)}")
print(classification_report(y_true, y_pred_rf, digits=4, output_dict=False))
print(confusion_matrix(y_true, y_pred_rf))
print("\n")

print(f"LLM time taken: {sum(elapsed_times_llm)/len(X_test)}\n")
print(classification_report(["attack" if y else "normal" for y in y_true], y_pred_llm, digits=4, output_dict=False))
print(confusion_matrix(["attack" if y else "normal" for y in y_true], y_pred_llm))

DT time taken: 0.00010290139007686524
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    221490
           1     0.9999    1.0000    1.0000     17403

    accuracy                         1.0000    238893
   macro avg     1.0000    1.0000    1.0000    238893
weighted avg     1.0000    1.0000    1.0000    238893

[[221489      1]
 [     0  17403]]


RF time taken: 0.0019271470386483267
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    221490
           1     1.0000    0.9999    1.0000     17403

    accuracy                         1.0000    238893
   macro avg     1.0000    1.0000    1.0000    238893
weighted avg     1.0000    1.0000    1.0000    238893

[[221490      0]
 [     1  17402]]


LLM time taken: 4.296412716681614e-05

              precision    recall  f1-score   support

      attack     0.7768    0.9629    0.8599     17403
      normal     0.9970    0.9783    0.9876    22